# HPC for AI - Part 2: fine-tuning an LLM on Isambard-AI

In this notebook you'll fine-tune the Mistral 7B language model on one GPU on the Isambard-AI supercomputer, so that it learns to follow instructions. Then you'll use it to label about 30,000 UK news headlines.

**How to use this notebook**

- Run the cells in order with **Shift + Enter**. Wait for each cell to finish before you run the next one: the `[*]` to the left of the cell changes to a number when it's done.
- The kernel name in the top right should say **Fine-tuning + Nsight**. If it doesn't, click on it and choose that kernel.
- Later sections use models and variables from earlier ones, so don't skip cells.
- The course notes go through each step in more detail, and have the exercises. The section numbers here match the notes.

**Data:** [Stanford Alpaca](https://github.com/tatsu-lab/stanford_alpaca) (non-commercial use), and UK front-page headlines from [The Hear](https://www.thehear.org).

## 1. Setup

### Load the libraries

This loads the libraries we'll use and sets where your results go. Set `MODE` to `"qlora"` or `"lora"`: you'll be told which to use, and section 3 explains the difference.

In [ ]:
import gc, glob, json, os, random, time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODE = "qlora"  # "qlora" or "lora"
MODEL_ID = "mistralai/Mistral-7B-v0.3"
PROJECTDIR = os.environ["PROJECTDIR"]
COURSE = f"{PROJECTDIR}/course"
WORK_PATH = os.environ["WORK_PATH"]
OUTPUT_DIR = f"{WORK_PATH}/sft-output/{MODE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

`free_gpu()` returns memory to the GPU after a model is deleted. We use it each time one model makes way for the next.

In [ ]:
def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

### Where things live

The model and datasets are in a shared course folder, `$PROJECTDIR/course`, that the instructors have set up. You can read them, but you can't change them.

Anything you create goes in your own folder, `WORK_PATH`, which is `$PROJECTDIR/$USER/llm-fine-tuning`.

## 2. What is fine-tuning?

A base language model is only trained to predict the next token, so it continues text rather than answering. Fine-tuning trains it a little further on a small, specific dataset.

### Tokenizer

The tokenizer turns text into the token IDs the model reads. Mistral 7B has no padding token, so we reuse the end-of-sequence token.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

### Dataset

Stanford Alpaca has about 52,000 instruction-following examples, each a single `text` field: an instruction, then `### Response:`, then the answer. This copy was saved to the course folder before the session.

In [ ]:
dataset = load_dataset("parquet", data_files=f"{COURSE}/alpaca/train.parquet", split="train")

print(len(dataset), "examples")
print(dataset[0]["text"])

### What can the base model do?

Give it the start of a sentence. What comes back is a plausible continuation, which is not necessarily the fact you were hoping for.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda")


def complete(model, text, max_new_tokens=40):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print(complete(base_model, "The capital of France is"))

Now ask a direct question. A base model often treats it as the start of a document, so it may write more questions or repeat itself.

In [ ]:
print(complete(base_model, "What is the capital of France?", max_new_tokens=80))

Now put questions into the layout that fine-tuning will teach. Look at how each answer **ends**: does it stop, or carry on and write a new `### Instruction:` of its own? We'll ask the same questions after training.

In [ ]:
QUESTIONS = [
    "Give three tips for staying healthy.",
    "Explain what a GPU is to a ten-year-old.",
    "Suggest a name for a bakery in Bristol and explain your choice.",
    "List the planets of the solar system in order from the Sun.",
]
TEMPLATE = ("Below is an instruction that describes a task. "
            "Write a response that appropriately completes the request.\n\n"
            "### Instruction:\n{question}\n\n### Response:\n")


def ask(model, question):
    return complete(model, TEMPLATE.format(question=question), max_new_tokens=150)

In [ ]:
for question in QUESTIONS:
    print("Q:", question, "\nBASE MODEL:", ask(base_model, question), "\n" + "-" * 60)

del base_model
free_gpu()

## 3. Training with LoRA and QLoRA

Fine-tuning every weight of a 7-billion-parameter model needs about 117 GB, more than the 96 GB on this GPU. **LoRA** freezes the model and trains small adapter matrices beside each layer. **QLoRA** does the same, and also stores the frozen weights in 4 bits.

### Training settings

The next three cells only define settings. The comments explain each one.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # the 4-bit format
    bnb_4bit_compute_dtype="bfloat16",     # unpack to this for every multiplication
    bnb_4bit_use_double_quant=True,        # also compress the format's scaling constants
)

if MODE == "qlora":
    model_init_kwargs = {"quantization_config": quantization_config, "dtype": "bfloat16"}
else:
    model_init_kwargs = {"dtype": "bfloat16"}

In [ ]:
lora_config = LoraConfig(
    r=16,                         # rank: the size of the small matrices
    lora_alpha=32,                # how strongly the adapter's output is added
    lora_dropout=0.05,            # drop 5% of adapter inputs, to reduce overfitting
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # put an adapter on every linear layer
)

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR, report_to="none",
    dataset_text_field="text", max_length=1024,
    packing=False,                                       # packing needs Flash Attention
    max_steps=200,                                       # 200 × 16 = 3,200 examples, about 6% of Alpaca
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,                       # each update uses 4 × 4 = 16 examples
    gradient_checkpointing=True,                         # recompute, not store: less memory, more compute
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10,
    logging_steps=10, save_steps=100, bf16=True,
    model_init_kwargs=model_init_kwargs,
)

### Train

`SFTTrainer` loads the model, adds the LoRA adapters and tokenises the dataset. The printed line shows how few parameters train.

In [ ]:
trainer = SFTTrainer(model=MODEL_ID, args=training_args, train_dataset=dataset,
                     processing_class=tokenizer, peft_config=lora_config)
trainer.model.print_trainable_parameters()

Training takes 8 to 9 minutes. The loss is printed every 10 steps and should fall.

Doing the optional profiling section? Training is the only chance to record a slice of it: see section 9 before you run this cell.

In [ ]:
torch.cuda.reset_peak_memory_stats()
with torch.cuda.nvtx.range("train"):
    result = trainer.train()

stats = {
    "mode": MODE,
    "train_minutes": round(result.metrics["train_runtime"] / 60, 1),
    "samples_per_second": round(result.metrics["train_samples_per_second"], 2),
    "peak_memory_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 1),
    "final_loss": round(result.training_loss, 3),
}
with open(f"{OUTPUT_DIR}/stats.json", "w") as f:
    json.dump(stats, f)
stats

### Save the adapter

This saves only the small adapter matrices, about 84 MB, not a copy of the model.

**Behind, or training failed?** Set `USE_FALLBACK = True` in this cell, and carry on with a trained adapter we prepared earlier.

In [ ]:
USE_FALLBACK = False  # True if you are behind or training failed

if USE_FALLBACK:
    ADAPTER = f"{COURSE}/fallback/{MODE}/final"
else:
    ADAPTER = f"{OUTPUT_DIR}/final"
    trainer.save_model(ADAPTER)

trainer = None
free_gpu()

### LoRA vs QLoRA: the whole room's results

This reads everyone's `stats.json`, so it fills up as people finish. Run it again later for the final numbers.

In [ ]:
rows = []
for path in glob.glob(f"{PROJECTDIR}/*/llm-fine-tuning/sft-output/*lora/stats.json"):
    try:
        with open(path) as f:
            rows.append(json.load(f))
    except (OSError, ValueError):
        pass  # someone's file is unreadable or half-written

results = pd.DataFrame(rows)
results.groupby("mode").mean().round(2).assign(people=results["mode"].value_counts())

### Why is QLoRA slower?

**What we measured** (yours will differ a little):

| | LoRA | QLoRA |
|---|---|---|
| Training time | 8.0 min | 8.8 min |
| Peak GPU memory | 16.1 GiB | 5.8 GiB |
| Final loss | 0.903 | 0.908 |

A profile of 20 seconds of each run shows why. QLoRA's longest-running GPU kernel is `kDequantizeBlockwise`, which unpacks the 4-bit weights back to 16 bits before every multiplication: 18% of all kernel time. LoRA doesn't run it at all. So QLoRA pays about 10% more time for about 64% less memory, and both learn equally well.

## 4. Did it work?

Load the base model and attach your adapter. `disable_adapter()` switches the adapter off for a moment, so the same model gives both answers. The first question was in the training data; the others weren't.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(model, ADAPTER)

This uses Transformers' `generate()`, one question and one token at a time. The cell times it, to compare with the next section.

In [ ]:
start = time.perf_counter()
with torch.cuda.nvtx.range("generate-hf"):
    for question in QUESTIONS:
        with model.disable_adapter():
            print("Q:", question, "\nBASE MODEL:", ask(model, question))
        print("\nFINE-TUNED MODEL:", ask(model, question), "\n" + "-" * 60)
hf_seconds = time.perf_counter() - start
print(f"Transformers generate(): {hf_seconds:.1f} s for {2 * len(QUESTIONS)} answers")

## 5. Faster inference with vLLM

vLLM is an inference engine that works on all the questions at once (**batching**) and replays recorded GPU work (**CUDA graphs**) instead of launching every kernel from Python.

Start it now. It takes a minute or two the first time, so run this cell just before the break.

In [ ]:
del model
free_gpu()

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(model=MODEL_ID, enable_lora=True, max_lora_rank=16,
          max_model_len=1024, gpu_memory_utilization=0.8)
adapter = LoRARequest(MODE, 1, ADAPTER)

**Break.** Take 15 minutes once the cell above has finished.

Now the same questions. `temperature=0` always picks the most likely next token, so the answers should closely match the Transformers ones.

In [ ]:
settings = SamplingParams(temperature=0, max_tokens=150)
prompts = [TEMPLATE.format(question=q) for q in QUESTIONS]

start = time.perf_counter()
with torch.cuda.nvtx.range("generate-vllm"):
    base_answers = llm.generate(prompts, settings)
    tuned_answers = llm.generate(prompts, settings, lora_request=adapter)
vllm_seconds = time.perf_counter() - start

for question, tuned in zip(QUESTIONS, tuned_answers):
    print("Q:", question, "\nFINE-TUNED MODEL:", tuned.outputs[0].text, "\n" + "-" * 60)
print(f"Transformers: {hf_seconds:.1f} s   vLLM: {vllm_seconds:.1f} s")

## 6. Prompting from a program

### One instruction over many inputs

`run` asks one instruction about each input in a list. Give it the whole list at once, so vLLM can batch it. Pass `lora_request=None` to use the base model instead of your adapter.

In [ ]:
TEMPLATE_WITH_INPUT = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n")
loop_settings = SamplingParams(temperature=0, max_tokens=150, stop=["###"])


def run(instruction, inputs, settings=loop_settings, lora_request=adapter):
    """Ask one instruction for each input, all in one batch."""
    prompts = [TEMPLATE_WITH_INPUT.format(instruction=instruction, input=text) for text in inputs]
    outputs = llm.generate(prompts, settings, lora_request=lora_request)
    return [out.outputs[0].text.strip() for out in outputs]

Some made-up reviews to try it on.

In [ ]:
reviews = [
    "The battery lasts all day and the screen is lovely.",
    "Stopped working after a week. Support never replied.",
    "It does the job. Nothing special.",
    "Absolutely brilliant, I've already bought a second one.",
    "Arrived late and the box was damaged, but the product itself is fine.",
    "Too expensive for what you get.",
]
SENTIMENT = "Classify the sentiment of this review as positive, negative or neutral. Answer with one word."

for review, label in zip(reviews, run(SENTIMENT, reviews)):
    print(f"{label!r:12} {review}")

### Check every answer, and retry the failures

`run_checked` cleans each answer, checks it against the answers you allow, retries only the failures with a little randomness, and returns `None` if it still fails.

In [ ]:
retry_settings = SamplingParams(temperature=0.7, max_tokens=20, stop=["###"])


def clean(answer):
    words = answer.lower().split()
    return words[0].strip(".,!:;\"'") if words else ""

In [ ]:
def run_checked(instruction, inputs, allowed, settings=loop_settings, lora_request=adapter):
    answers = [clean(a) for a in run(instruction, inputs, settings, lora_request)]
    for attempt in range(3):
        failed = [i for i, a in enumerate(answers) if a not in allowed]
        if failed:
            print(f"Retrying {len(failed)} answer(s)")
            for i, a in zip(failed, run(instruction, [inputs[i] for i in failed], retry_settings, lora_request)):
                answers[i] = clean(a)
    return [a if a in allowed else None for a in answers]

In [ ]:
labels = run_checked(SENTIMENT, reviews, {"positive", "negative", "neutral"})

table = pd.DataFrame({"review": reviews, "sentiment": labels})
table.to_csv(f"{OUTPUT_DIR}/sentiment.csv", index=False)
table

## 7. A research question

Has each UK news outlet given more or less of its front page to the environment over the last two years? The data is the lead headline on each UK outlet's front page, sampled every 6 hours from September 2024 to October 2026 from [The Hear](https://www.thehear.org).

In [ ]:
with open(f"{COURSE}/headlines/uk_headlines.json") as f:
    data = json.load(f)

rows = [(paper, int(year), title, info["subtitle"])
        for paper, years in data.items()
        for year, titles in years.items()
        for title, info in titles.items()]

print(len(rows), "headlines from", len(data), "outlets")
for paper, year, title, subtitle in random.Random(0).sample(rows, 5):
    print(f"{paper} ({year}): {title}")

### One at a time

First, the obvious way: a function that classifies one headline, called once per headline. We only try it on 50.

In [ ]:
CLASSIFY = ("Is this news headline about the environment, for example climate change, extreme weather, "
            "energy, pollution, wildlife or nature? Answer yes or no.")
classify_settings = SamplingParams(temperature=0, max_tokens=5, stop=["###"])
inputs = [f"{title}\n{subtitle}" if subtitle else title for _, _, title, subtitle in rows]


def is_about_environment(text):
    prompt = TEMPLATE_WITH_INPUT.format(instruction=CLASSIFY, input=text)
    output = llm.generate([prompt], classify_settings, lora_request=adapter, use_tqdm=False)
    return clean(output[0].outputs[0].text) == "yes"

In [ ]:
start = time.perf_counter()
with torch.cuda.nvtx.range("classify-loop"):
    loop_labels = [is_about_environment(text) for text in inputs[:50]]
loop_seconds = time.perf_counter() - start

print(f"One at a time: {loop_seconds:.1f} s for 50 headlines, {20 * loop_seconds:.0f} ms each")
print(f"All {len(rows)} would take about {loop_seconds / 50 * len(rows) / 60:.0f} minutes")

### All at once

Now classify the whole list in one call. Compare the time per headline with the loop.

In [ ]:
start = time.perf_counter()
with torch.cuda.nvtx.range("classify-batch"):
    labels = run_checked(CLASSIFY, inputs, {"yes", "no"}, classify_settings)
batch_seconds = time.perf_counter() - start

print(f"All at once: {batch_seconds:.1f} s for {len(rows)} headlines, "
      f"{1000 * batch_seconds / len(rows):.1f} ms each")
print("Unanswered:", labels.count(None))
print("Same answer as the loop:", sum(a == (b == "yes") for a, b in zip(loop_labels, labels)), "of 50")

### Count and plot

Count each outlet's headlines per year, and the share about the environment. An outlet with no headlines in a year shows `NaN`.

In [ ]:
counts = pd.DataFrame(0, index=sorted({r[1] for r in rows}), columns=sorted(data))
env_counts = counts.copy()

for (paper, year, _, _), label in zip(rows, labels):
    counts.loc[year, paper] += 1
    if label == "yes":
        env_counts.loc[year, paper] += 1

share = (100 * env_counts / counts).round(1)
share.to_csv(f"{OUTPUT_DIR}/environment_share.csv")
share.T

In [ ]:
big = counts.columns[counts.sum() >= 100]  # too few headlines makes a percentage meaningless
order = share[big].mean().sort_values().index

ax = share[order].T.plot.barh(figsize=(8, 0.4 * len(order) + 1), width=0.8)
ax.set_xlabel("% of lead headlines about the environment")
ax.legend(title="Year")
plt.tight_layout()
plt.show()

Check by eye: are these really about the environment?

In [ ]:
yes = [(paper, title) for (paper, _, title, _), label in zip(rows, labels) if label == "yes"]
print(len(yes), "headlines labelled yes. A random 15:")
for paper, title in random.Random(0).sample(yes, min(15, len(yes))):
    print(f"  {paper:22} {title}")

## 8. Wrapping up

### How far can we trust this?

- **The classifier is the weak link.** The adapter learned a question-and-answer format; it was never taught what counts as "the environment".
- **Lead stories only.** This measures what each outlet chose to lead with, not everything it published.
- **Partial years.** 2024 starts in September and 2026 ends in October.
- **Sampling.** One snapshot every 6 hours misses lead stories that came and went in between.

Your results are in `WORK_PATH`, which isn't backed up, so copy anything you want to keep.

## 9. Optional: profiling

This section is optional, and the course notes explain it. Run these commands in a terminal (**File → New → Terminal**), not in the notebook. Set up the terminal and copy in the profiles we recorded earlier:

```bash
module load cudatoolkit/24.11_12.6
W=$PROJECTDIR/$USER/llm-fine-tuning
cp -n $PROJECTDIR/course/fallback/reports/*.nsys-rep $W/
```

To record your own, start recording, run a cell, then stop. Training (section 3) is named `slice-qlora` or `slice-lora`; stop it after about 20 seconds.

```bash
nsys start --session=nb-$USER -o $W/generate-hf --force-overwrite=true
nsys stop --session=nb-$USER
```

Summarise a report:

```bash
bash $PROJECTDIR/course/kernels/profile-summary.sh generate-hf
```

## Before you go

Free the GPU memory, and save this notebook (**File → Save Notebook**).

In [ ]:
del llm
free_gpu()

Your session keeps the GPU reserved until it runs out of time, even if you aren't using it. When you've finished, go to **File → Hub Control Panel** and click **Stop My Server**. This lets other people use the GPU.